# 충남대 NLP 챗봇 — Colab 모두 실행(Run all) 전용 노트북

**런타임을 T4 GPU 로 바꿔주세요** (런타임 → 런타임 유형 변경 → T4 GPU).

**사전 준비 (딱 1개):** 구글 드라이브 `MyDrive` 에 `chroma_db_updated_20260609.tar.gz` 파일을 업로드해 두세요. (이미 올려두셨다면 그대로 두면 됩니다.)

그다음 **런타임 → 모두 실행** 한 번이면 끝입니다. 위에서 아래로 멈추지 않고 전부 돌아갑니다.

첫 실행은 분류 모델 학습 때문에 약 **20~35분** 걸립니다. (드라이브에 모델이 한 번 저장되면 다음 모두 실행부터는 자동 복원되어 훨씬 빨라집니다.)

마지막 셀은 데모 UI 라서 일부러 **계속 떠 있습니다(정상)**. 공개 링크로 시연·녹화한 뒤 셀 정지(■) 버튼으로 멈추면 됩니다.

In [ ]:
!nvidia-smi

## 1. 코드 받기 + 라이브러리 설치

저장소를 clone 하고 해당 폴더로 이동한 뒤 의존성을 설치합니다. (이미 clone 되어 있으면 git clone 은 건너뛰고 그 폴더로 들어갑니다 — 멱등)

In [ ]:
import os, subprocess
if not os.path.isdir('cnu-llm-bot'):
    subprocess.run(['git', 'clone',
                    'https://github.com/Longarden/cnu-llm-bot.git'], check=True)
else:
    print('[git] cnu-llm-bot 폴더가 이미 있음 → clone 건너뜀')
%cd cnu-llm-bot
!pip install -q -r requirements.txt
print('[setup] 설치 완료, 작업 폴더:', os.getcwd())

## 2. 구글 드라이브 마운트

모델/벡터DB 자산을 드라이브에서 자동으로 가져오거나, 새로 만든 뒤 드라이브에 저장하기 위해 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 분류 모델 준비 (드라이브 자동 연동)

드라이브에 모델이 있으면 복원해서 바로 사용하고, 없으면 학습(약 10~20분) 후 드라이브에 저장합니다. 다음 모두 실행부터는 자동 복원되어 빠릅니다.

In [ ]:
import os, shutil, subprocess
DRIVE_MODEL = '/content/drive/MyDrive/cnu_model'
if os.path.exists(os.path.join(DRIVE_MODEL, 'config.json')):
    shutil.rmtree('model', ignore_errors=True)
    shutil.copytree(DRIVE_MODEL, 'model')
    print('[model] 드라이브에서 복원해 사용:', DRIVE_MODEL)
else:
    print('[model] 드라이브에 없음 → roberta-large 학습(10~20분)')
    env = {**os.environ, 'CLS_MODEL': 'klue/roberta-large', 'CLS_MAX_LEN': '128', 'CLS_EPOCHS': '6'}
    subprocess.run(['python', 'scripts/train_classifier.py'], env=env, check=True)
    shutil.rmtree(DRIVE_MODEL, ignore_errors=True)
    shutil.copytree('model', DRIVE_MODEL)
    print('[model] 학습 완료 + 드라이브 저장:', DRIVE_MODEL, '(다음 모두실행부터는 자동 복원=빠름)')
import json
c = json.load(open('model/config.json'))
print('[model] 백본:', c.get('architectures'), c.get('_name_or_path'), 'hidden=', c.get('hidden_size'))

## 4. 벡터DB(chroma) 준비 (드라이브 tar 자동 연동)

로컬에 chroma_db 가 있으면 그대로 쓰고, 없으면 드라이브 tar 에서 복원합니다. tar 도 없으면 새로 빌드(약 15분) 후 드라이브에 저장합니다.

**중요:** `embedding/vector_store.py` 가 프로젝트 루트의 `chroma_db/` 를 하드코딩으로 사용하므로, tar 는 반드시 루트에 `chroma_db/` 가 생기도록 풀립니다.

In [ ]:
import os, glob, subprocess
DRIVE_TAR = '/content/drive/MyDrive/chroma_db_updated_20260609.tar.gz'
if glob.glob('chroma_db/*.sqlite3'):
    print('[chroma] 로컬에 이미 있음 → 그대로 사용')
elif os.path.exists(DRIVE_TAR):
    subprocess.run(['tar', 'xzf', DRIVE_TAR, '-C', '.'], check=True)
    print('[chroma] 드라이브 tar에서 복원:', DRIVE_TAR)
else:
    print('[chroma] 드라이브 tar 없음 → 새로 빌드(약 15분)')
    subprocess.run(['python', 'scripts/rebuild_index.py'], check=True)
    subprocess.run(['tar', 'czf', '/content/drive/MyDrive/chroma_db_updated_20260609.tar.gz', 'chroma_db'])
    print('[chroma] 빌드 완료 + 드라이브 저장')
import chromadb
cl = chromadb.PersistentClient(path='chroma_db')
print('[chroma] 컬렉션:', [ (c.name, c.count()) for c in cl.list_collections() ])

## 5. Task1 — 질문유형 분류 추론

`src/classifier.py` 로 `outputs/cls_output.json` 을 만들고 제출 포맷을 검증합니다 (리스트 / 각 항목 id·question·label / label 은 0~4 정수).

In [ ]:
import subprocess, json
subprocess.run(['python', 'src/classifier.py'], check=True)
with open('outputs/cls_output.json', encoding='utf-8') as f:
    cls = json.load(f)
assert isinstance(cls, list) and len(cls) > 0, '[Task1] cls_output.json 이 비어있거나 리스트가 아닙니다.'
for row in cls:
    assert 'id' in row and 'question' in row and 'label' in row, \
        f'[Task1] 각 항목에 id·question·label 키가 모두 있어야 합니다: {row}'
    assert isinstance(row['label'], int) and 0 <= row['label'] <= 4, \
        f"[Task1] label 은 0~4 사이 정수여야 합니다: {row.get('label')}"
from collections import Counter
LBL = {0: '졸업요건', 1: '학교공지', 2: '학사일정', 3: '식단', 4: '통학/셔틀'}
dist = Counter(r['label'] for r in cls)
print('[Task1] OK — 총', len(cls), '건, 라벨분포:', dict(sorted(dist.items())))
print('--- 디버깅: 질문별 예측 라벨 ---')
for row in cls:
    print(f"[{row['id']}] label={row['label']}({LBL.get(row['label'], '?')})  Q: {row['question']}")

## 6. Task2 — 배치 챗봇 추론

`src/gen_chat_output.py` 로 `outputs/chat_output.json` 을 만들고 검증합니다 (각 항목 id·user·model / model 답변이 빈문자열·None·'생성 오류' 가 아니어야 함).

In [ ]:
import subprocess, json, os
# 전체 문항 처리(디버깅 — 모든 답 확인). 채점자도 chatbot.sh 로 전체 처리.
subprocess.run(['python', 'src/gen_chat_output.py'], check=True)
with open('outputs/chat_output.json', encoding='utf-8') as f:
    chat = json.load(f)
assert isinstance(chat, list) and len(chat) > 0, '[Task2] chat_output.json 이 비어있거나 리스트가 아닙니다.'
for row in chat:
    assert 'id' in row and 'user' in row and 'model' in row, \
        f'[Task2] 각 항목에 id·user·model 키가 모두 있어야 합니다: {row}'
    m = row['model']
    assert m is not None and isinstance(m, str) and m.strip() != '', \
        f"[Task2] model 답변이 비어있습니다: {row.get('id')}"
    assert not m.startswith('생성 오류'), f"[Task2] 생성 오류 답변이 있습니다: {row.get('id')} → {m[:80]}"
print('[Task2] OK — 총', len(chat), '건, 모두 정상 답변')
print('=== 디버깅: 전체 질문 → 답변 ===')
for row in chat:
    print(f"\n[{row['id']}] Q: {row['user']}\n    A: {row['model']}")

## 7. 휘발성(live) vs 정적(static) 라우팅 점검

질문 4개의 라우팅 소스를 확인합니다. 오늘 학식·다음주 셔틀 → `live`, 셔틀 노선·졸업 학점 → `static` 이 기대값입니다. (이 셀은 실패해도 try/except 로 감싸 다음 셀로 넘어갑니다.)

In [ ]:
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
try:
    from src.chat_pipeline import chat_answer
    probes = [
        ('오늘 학식 뭐 나와요?', 'live'),
        ('셔틀 노선 알려줘', 'static'),
        ('다음주 셔틀 정상 운행하나요?', 'live'),
        ('졸업까지 몇 학점이에요?', 'static'),
    ]
    print('| 질문 | 기대 | 실제 source |')
    print('|---|---|---|')
    for q, expect in probes:
        try:
            meta = chat_answer(q, return_meta=True)[1]
            src = meta.get('source', '?')
        except Exception as e:
            src = f'오류:{e}'
        mark = 'OK' if src == expect else 'CHECK'
        print(f'| {q} | {expect} | {src} ({mark}) |')
except Exception as e:
    print('[라우팅 점검] 건너뜀(다음 셀 계속 진행):', e)

## 8. Task3 — 실시간 반영(옵션)

`src/realtime_model.py` 로 `outputs/realtime_output.json` 을 만들고 검증합니다 (각 항목 id·user·model / model 답변이 비어있지 않아야 함).

In [ ]:
import subprocess, json, os
# 전체 문항 처리(디버깅). 채점자도 chatbot.sh 로 전체 처리.
subprocess.run(['python', 'src/realtime_model.py'], check=True)
with open('outputs/realtime_output.json', encoding='utf-8') as f:
    rt = json.load(f)
assert isinstance(rt, list) and len(rt) > 0, '[Task3] realtime_output.json 이 비어있거나 리스트가 아닙니다.'
for row in rt:
    assert 'id' in row and 'user' in row and 'model' in row, \
        f'[Task3] 각 항목에 id·user·model 키가 모두 있어야 합니다: {row}'
    m = row['model']
    assert m is not None and isinstance(m, str) and m.strip() != '', \
        f"[Task3] model 답변이 비어있습니다: {row.get('id')}"
print('[Task3] OK — 총', len(rt), '건')
print('=== 디버깅: 전체 질문 → 답변 ===')
for row in rt:
    print(f"\n[{row['id']}] Q: {row['user']}\n    A: {row['model']}")

## 9. 제출물 패키징

`scripts/package_submission.sh` 로 제출 zip 을 만듭니다 (모델 가중치 포함, 제출자 이름 = 장정원). 결과는 `dist/Termproject_장정원.zip` 입니다.

In [ ]:
import subprocess, os
env = {**os.environ, 'INCLUDE_MODEL': '1', 'NAME': '장정원'}
subprocess.run(['bash', 'scripts/package_submission.sh'], env=env, check=True)
zip_path = 'dist/Termproject_장정원.zip'
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'[제출] {zip_path} — {size_mb:.1f} MB')
print('[안내] 용량이 너무 크면(예: 100MB 이상) 모델 가중치는 zip 에서 빼고 드라이브 다운로드 링크로 제출하세요.')

## 10. UI 미리보기 (수동 실행용 — 모두 실행은 건너뜀)

화면 레이아웃만 빠르게 확인하고 싶다면, **새 셀을 직접 만들어** 아래 명령을 붙여 실행하세요. (모델 없이 UI 만 띄우며, 블로킹이라 모두 실행에는 넣지 않습니다.)

```
!UI_MOCK=1 python src/chatbot_ui.py
```

## 11. ★ 실제 UI 데모 (마지막 셀 — 계속 떠 있는 게 정상)

커스텀 FastAPI UI 를 cloudflared 공개 링크로 띄웁니다. **이 셀은 계속 실행 상태로 떠 있습니다(정상).**

1. 출력에 나오는 `https://...trycloudflare.com` 링크를 클릭
2. 5개 유형 질문을 차례로 물어보며 시연
3. **Win + Alt + R** 로 화면 녹화 2분
4. 끝나면 이 셀의 **정지(■)** 버튼을 눌러 종료

In [ ]:
!pip -q install fastapi uvicorn starlette
# 데모용: 커스텀 UI 바로 띄움(2.4B 기본 + 워밍업이라 첫 질문도 빠름). 채점자는 chatbot.sh(전체 배치+UI) 실행.
!GRADIO_SHARE=1 python src/chatbot_ui.py

## 제출 체크리스트

- [ ] 제출 zip 다운로드: 좌측 파일 탭 → `dist/Termproject_장정원.zip` 우클릭 → 다운로드
- [ ] 시연 영상 2분 (5개 유형 질문 시연)
- [ ] 발표 5분 준비
- [ ] **마감: 6/12**